# H-Bar Symbolic - Kaggle Execution Notebook

This notebook executes the H-Bar pilot study (N=15) on Kaggle GPUs to verify the H-Bar effect before scaling to the full N=120 pre-registered runs.

## Objectives

1. **Verify H-Bar Effect:** Confirm OOD accuracy >90% and σ̂_A >0.9
2. **Compare Conditions:** Test both Additive (B) and Multiplicative (C) loss coupling
3. **Check Stability:** Ensure training doesn't diverge (especially for multiplicative)
4. **Phase 2 Detection:** Verify crystallization occurs within first 1,000 steps

## Expected Outcomes

| Metric | Baseline | H-Bar Target |
|--------|----------|--------------|
| ID Accuracy | 91.9% | >90% |
| OOD Accuracy | 63.0% | >90% |
| σ̂_A | 0.685 | >0.9 |
| GCA (g_A) | -0.02 | Positive |
| Phase 2 Entry | Never | <1,000 steps |

## Estimated Runtime

- Per run: ~15 minutes (based on baseline)
- N=15 runs: ~4 hours total
- Kaggle session limit: 12 hours ✓

## Step 1: Environment Setup

First, we'll clone the repository and install dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/basyirin-dev/hbar-symbolic.git
%cd hbar-symbolic

# List files to verify
!ls -la

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

# Install the package
!pip install -q -e .

In [ ]:
# Verify JAX installation and GPU availability
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

## Step 2: Mount Baseline Dataset

Mount the baseline results dataset for comparison with H-Bar trajectories.

In [ ]:
# Check for baseline dataset
!ls -la /kaggle/input/datasets/basyirinamsyar/baseline-metrics-and-model-params/

# Copy baseline files for comparison
!cp /kaggle/input/datasets/basyirinamsyar/baseline-metrics-and-model-params/baseline_metrics.csv ./baseline_metrics.csv 2>/dev/null || echo 'Baseline metrics not found, using existing'
!cp /kaggle/input/datasets/basyirinamsyar/baseline-metrics-and-model-params/model_params.msgpack ./model_params.msgpack 2>/dev/null || echo 'Model params not found, using existing'

print("Baseline files copied successfully")

## Step 3: Verify Data Files

Ensure the frozen evaluation datasets are present.

In [ ]:
# Check data directory
!ls -la data/

# Verify file sizes
import os
for f in ['scan_id_eval.json', 'scan_ood_eval.json', 'cogs_id_eval.json', 'cogs_ood_eval.json']:
    path = os.path.join('data', f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"{f}: {size:,} bytes")
    else:
        print(f"WARNING: {f} not found!")

## Step 4: Run Pilot Study - Condition C (Multiplicative)

Execute N=15 runs with multiplicative loss coupling (primary H-Bar intervention).

In [ ]:
# Run multiplicative condition (Condition C) - 15 runs
!python scripts/train_hbar.py \
    --domain scan \
    --condition multiplicative \
    --n_runs 15 \
    --batch-size 64 \
    --total-steps 5000 \
    --eval-interval 500 \
    --learning-rate 0.001 \
    --lambda-sigma 0.5 \
    --seed 42 \
    --output-dir .

## Step 5: Run Pilot Study - Condition B (Additive)

Execute N=15 runs with additive loss coupling (stable comparison).

In [ ]:
# Run additive condition (Condition B) - 15 runs
!python scripts/train_hbar.py \
    --domain scan \
    --condition additive \
    --n_runs 15 \
    --batch-size 64 \
    --total-steps 5000 \
    --eval-interval 500 \
    --learning-rate 0.001 \
    --lambda-sigma 0.5 \
    --seed 42 \
    --output-dir .

## Step 6: Analyze Pilot Results

Load and analyze the pilot results to verify the H-Bar effect.

In [ ]:
import pandas as pd
import numpy as np

# Load pilot summary
summary = pd.read_csv('pilot_results_summary.csv')
print("Pilot Results Summary:")
print(summary.head(10))

In [ ]:
# Compute statistics by condition
print("\n" + "="*60)
print("PILOT STUDY RESULTS BY CONDITION")
print("="*60)

for condition in ['additive', 'multiplicative']:
    cond_data = summary[summary['condition'] == condition]
    print(f"\n--- {condition.upper()} Condition (N={len(cond_data)}) ---")
    print(f"OOD Accuracy:  {cond_data['final_ood_accuracy'].mean():.4f} ± {cond_data['final_ood_accuracy'].std():.4f}")
    print(f"ID Accuracy:   {cond_data['final_id_accuracy'].mean():.4f} ± {cond_data['final_id_accuracy'].std():.4f}")
    print(f"σ̂_A:           {cond_data['final_sigma_hat'].mean():.4f} ± {cond_data['final_sigma_hat'].std():.4f}")
    
    # Phase 2 entry detection
    phase2_count = cond_data['phase2_entry_step'].notna().sum()
    print(f"Phase 2 Entry: {phase2_count}/{len(cond_data)} runs")
    if phase2_count > 0:
        print(f"Mean entry step: {cond_data['phase2_entry_step'].mean():.1f}")
    
    # H-Bar effect verification
    ood_above_90 = (cond_data['final_ood_accuracy'] > 0.9).sum()
    sigma_above_90 = (cond_data['final_sigma_hat'] > 0.9).sum()
    print(f"OOD > 90%: {ood_above_90}/{len(cond_data)} runs")
    print(f"σ̂_A > 0.9: {sigma_above_90}/{len(cond_data)} runs")

In [ ]:
# Compare with baseline
baseline_ood = 0.63
baseline_sigma = 0.685

print("\n" + "="*60)
print("H-BAR EFFECT VERIFICATION")
print("="*60)
print(f"Baseline OOD Accuracy: {baseline_ood:.1%}")
print(f"Baseline σ̂_A:          {baseline_sigma:.3f}")
print()

for condition in ['additive', 'multiplicative']:
    cond_data = summary[summary['condition'] == condition]
    mean_ood = cond_data['final_ood_accuracy'].mean()
    mean_sigma = cond_data['final_sigma_hat'].mean()
    
    ood_improvement = mean_ood - baseline_ood
    sigma_improvement = mean_sigma - baseline_sigma
    
    print(f"\n{condition.upper()} Condition:")
    print(f"  OOD Accuracy:  {mean_ood:.4f} (Δ = {ood_improvement:+.4f})")
    print(f"  σ̂_A:           {mean_sigma:.4f} (Δ = {sigma_improvement:+.4f})")
    
    if mean_ood > 0.9 and mean_sigma > 0.9:
        print(f"  ✅ H-BAR EFFECT CONFIRMED!")
    else:
        print(f"  ⚠️ H-Bar effect not fully achieved")

## Step 7: Visualize Training Trajectories

Plot the training metrics to observe the Phase 2 entry and crystallization dynamics.

In [ ]:
import matplotlib.pyplot as plt
import glob

# Find all metrics files
multiplicative_files = sorted(glob.glob('hbar_multiplicative_run_*_metrics.csv'))
additive_files = sorted(glob.glob('hbar_additive_run_*_metrics.csv'))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('H-Bar Training Trajectories', fontsize=16)

# Plot OOD Accuracy
ax = axes[0, 0]
for f in multiplicative_files[:5]:  # Plot first 5 runs
    df = pd.read_csv(f)
    ax.plot(df['step'], df['ood_accuracy'], alpha=0.5)
ax.axhline(y=0.9, color='green', linestyle='--', label='Target (90%)')
ax.set_title('OOD Accuracy (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy')
ax.legend()

# Plot σ̂_A
ax = axes[0, 1]
for f in multiplicative_files[:5]:
    df = pd.read_csv(f)
    ax.plot(df['step'], df['sigma_ode'], alpha=0.5)
ax.axhline(y=0.5, color='red', linestyle='--', label='σ_critical')
ax.axhline(y=0.9, color='green', linestyle='--', label='Target')
ax.set_title('σ̂_A Trajectory (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('σ̂_A')
ax.legend()

# Plot ID Accuracy
ax = axes[1, 0]
for f in multiplicative_files[:5]:
    df = pd.read_csv(f)
    ax.plot(df['step'], df['id_accuracy'], alpha=0.5)
ax.set_title('ID Accuracy (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy')

# Plot Compositional Penalty
ax = axes[1, 1]
for f in multiplicative_files[:5]:
    df = pd.read_csv(f)
    if 'compositional_penalty' in df.columns:
        ax.plot(df['step'], df['compositional_penalty'], alpha=0.5)
ax.set_title('Compositional Penalty (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('Penalty Weight')

plt.tight_layout()
plt.savefig('hbar_training_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Save Results for Phase 4

Package the results for download and further analysis.

In [ ]:
import shutil

# Create results directory
!mkdir -p hbar_pilot_results

# Copy all CSV files
!cp pilot_results_summary.csv hbar_pilot_results/
!cp hbar_*_metrics.csv hbar_pilot_results/

# Copy best model parameters (highest OOD accuracy)
best_multiplicative = summary[summary['condition'] == 'multiplicative']['final_ood_accuracy'].idxmax()
best_run_idx = summary.loc[best_multiplicative, 'run_idx']
best_params = f'model_params_hbar_multiplicative_run_{int(best_run_idx)}.msgpack'
!cp {best_params} hbar_pilot_results/best_model_params.msgpack

print(f"Best multiplicative run: {int(best_run_idx)} (OOD: {summary.loc[best_multiplicative, 'final_ood_accuracy']:.4f})")
print(f"Saved to: hbar_pilot_results/")

# List all output files
!ls -la hbar_pilot_results/

## Step 9: Next Steps

Based on the pilot results:

### If H-Bar Effect Confirmed (OOD > 90%, σ̂_A > 0.9):
1. Proceed to full N=120 pre-registered runs
2. Run on both SCAN and COGS domains
3. Submit results for publication

### If H-Bar Effect Not Confirmed:
1. Debug signal extraction pipeline
2. Adjust hyperparameters (λ_σ, κ_α)
3. Re-run pilot with modifications

### Commands for Full N=120 Run:
```bash
# Run in parallel (2 Kaggle sessions)
# Session 1: Runs 1-60
python scripts/train_hbar.py --domain scan --condition multiplicative --n_runs 60 --seed 42

# Session 2: Runs 61-120
python scripts/train_hbar.py --domain scan --condition multiplicative --n_runs 60 --seed 60042
```